# Challenger-guided AMP design: retrospective experiment

Use a GPU runtime for ESM embeddings. This notebook runs the **development experiment**, with separate train, calibration, validation, and test families. It does not retrain the peptide generator or claim experimental activity for new sequences.

First create `colab_bundle/challenger.zip` locally with `python scripts/bundle_challenger.py`, then upload it below. Your saved Drive directory preserves embeddings and fitted models across sessions. Use the same source ZIP, options, and dependency lock when resuming.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/amp-challenger/run-001')
WORK.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/amp-challenger')
ROOT.mkdir(parents=True, exist_ok=True)
print('Upload the locally generated challenger.zip')
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one ZIP bundle'
bundle_bytes = next(iter(uploaded.values()))
source_hash = hashlib.sha256(bundle_bytes).hexdigest()
marker = WORK / 'source_bundle_sha256.txt'
if marker.exists() and marker.read_text().strip() != source_hash:
    raise RuntimeError('Different source bundle: choose a fresh WORK directory')
zip_path = WORK / 'source_bundle.zip'
zip_path.write_bytes(bundle_bytes)
with zipfile.ZipFile(zip_path) as archive:
    for item in archive.infolist():
        destination = (ROOT / item.filename).resolve()
        if not destination.is_relative_to(ROOT.resolve()):
            raise ValueError('Unsafe ZIP member')
    archive.extractall(ROOT)
marker.write_text(source_hash + '\n')
os.environ['HF_HOME'] = str(WORK / 'huggingface_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Experiment directory:', WORK)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--group', 'research', '--python', sys.executable], cwd=ROOT, check=True)

def run_stage(stage, *extra):
    subprocess.run(['uv', 'run', '--no-sync', 'python', '-m',
                    'amp_challenge_2027.research', stage, '--work', str(WORK), *extra],
                   cwd=ROOT, check=True)

run_stage('prepare')


## Extract sequence embeddings once

The pinned 35M ESM-2 model uses CUDA if available. Atomic batch files allow resuming extraction after a disconnect. The notebook persists the model cache on Drive too. Do not change the batch size or model midway through a run.


In [ ]:
run_stage('embed', '--batch-size', '64')
EMBEDDINGS = WORK / 'embeddings/vectors.npz'


## Develop and compare

E. coli is the primary task; the other two species are prespecified secondary tasks. Models are small CPU regressors on cached features. The `descriptors` ablation omits ESM; the `esm` run includes it as a contrasting predictor. Both runs compare the potency recipe, ordinary ensemble mean, conservative ensemble, and learned challenger. The ESM run additionally includes ESM alone and representation-matched ensemble controls.

Validation tunes penalty strength. These are exploratory results. A selected penalty of zero means the warning provided no selection benefit under the tuning rule. Dataset, model, code, and configuration changes invalidate cache reuse.


In [ ]:
SPECIES = ['E. coli', 'S. aureus', 'P. aeruginosa']
RUN_DESCRIPTOR_ABLATION = True
for species in SPECIES:
    if RUN_DESCRIPTOR_ABLATION:
        run_stage('develop', '--species', species)
    run_stage('develop', '--species', species, '--embeddings', str(EMBEDDINGS))


In [ ]:
from IPython.display import Markdown, display
for report in sorted((WORK / 'results').glob('*/*/DEVELOPMENT.md')):
    display(Markdown(report.read_text()))
print('Full JSON reports, CSV predictions, and frozen policies are in:', WORK / 'results')


## Final test — leave locked until development is settled

The cell below is disabled by default. Review development comparisons and choose the approach before changing `UNLOCK_FINAL_TEST`. Test evaluation reuses the frozen policy and models; no refitting or tuning occurs. Once you inspect test outcomes, subsequent method changes cannot claim that same test as untouched.

This notebook cannot enforce scientific blinding outside its workflow. Choosing another seed or directory after inspecting test outcomes does not restore an untouched test set.


In [ ]:
UNLOCK_FINAL_TEST = False
if UNLOCK_FINAL_TEST:
    for species in SPECIES:
        run_stage('evaluate-test', '--species', species, '--embeddings', str(EMBEDDINGS))
else:
    print('Final test remains untouched. Review DEVELOPMENT.md first.')
